# 1. Trazabilidad de correcciones

Este capítulo responde al Bloque A de la guía: cada observación sobre el entregable anterior tiene una fila con la acción realizada, el lugar
del libro donde se ve y la evidencia. Al final se audita de forma automática que los cuadernos del libro se ejecutaron de corrido y sin
rutas absolutas, que fue la observación principal del docente en el primer entregable.

**Origen de las observaciones (declarado).** Al cerrar este capítulo no se había recibido retroalimentación escrita del docente sobre el
entregable 3 (el "Entregable 2" de la guía). Por eso la tabla reúne tres fuentes, marcadas en la columna `origen`:

1. **Docente, E1:** la retroalimentación escrita del primer entregable (`../correcciones.txt`). Sus observaciones aplican a todos los
   cuadernos posteriores.
2. **Guía E4, sec. 3:** la lista de puntos que la guía pide revisar del entregable anterior (ETL, EDA, validación anidada, optimización,
   comparación estadística, interpretabilidad, fuga de información).
3. **Autoauditoría:** problemas que encontró el equipo al revisar su propio entregable 3 y al construir el modelo original.

Si llega la retroalimentación del docente, se agrega una fila por observación con origen "Docente, E3".

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# El libro se ejecuta desde la raiz del repositorio (entregable3); no se usan rutas absolutas.
RAIZ = Path.cwd()
assert (RAIZ / "experiments" / "protocolo.md").is_file(), "Ejecute este cuaderno desde la raiz del repositorio"
pd.set_option("display.max_colwidth", 200)
print("Raiz del proyecto:", RAIZ.name)

Raiz del proyecto: entregable3


## 1.1 Tabla de trazabilidad

Columnas pedidas por la guía: observación, acción realizada, ubicación en el libro y evidencia. La tabla se guarda también como
`resultados_e4/trazabilidad_correcciones.csv` para la tabla maestra.

In [2]:
filas = [
    # --- Docente, E1 (correcciones.txt) ---
    ("Docente, E1", "Un output marcado como ejecucion 67 en una celda que hoy ocupa la posicion 55: no se puede certificar que las cifras salgan del codigo tal como esta.",
     "Todos los cuadernos se ejecutan de corrido en un kernel nuevo con un script (correr_capitulos.py, nbclient); ninguna cifra se escribe a mano.",
     "Cap. 1, sec. 1.3", "Auditoria automatica: contador de ejecucion 1, 2, ..., n en cada cuaderno (tabla 1.2)."),
    ("Docente, E1", "El cuaderno nunca se ejecuto de corrido en una sola sesion.",
     "Igual que la fila anterior; los calculos costosos se hacen con scripts versionados (experiments/) y los cuadernos leen sus resultados guardados.",
     "Cap. 1, sec. 1.3; Anexo de reproducibilidad", "Tabla 1.2; resultados_e4/*.json con fecha y versiones."),
    ("Docente, E1", "Rutas absolutas de al menos dos computadores distintos.",
     "Todas las rutas son relativas a la raiz del repositorio (Path.cwd() o config.RAIZ); en las salidas solo se imprime RAIZ.name.",
     "Cap. 1, sec. 1.3", "Tabla 1.2: conteo de rutas absolutas en codigo y salidas = 0."),
    ("Docente, E1", "Una celda cuyo output no corresponde al codigo mostrado.",
     "Ejecucion completa en kernel nuevo antes de compilar el libro; el libro no se compila con salidas viejas.",
     "Cap. 1, sec. 1.3", "Tabla 1.2 (orden de ejecucion consecutivo)."),
    # --- Guia E4, sec. 3 ---
    ("Guia E4, sec. 3", "ETL: transformaciones con parametros aprendidos ajustadas solo dentro de cada pliegue de entrenamiento.",
     "El preprocesador (imputacion, escala, one-hot) se construye sin ajustar y se ajusta con el entrenamiento de cada pliegue; la variable de vecindario usa codificacion fuera de pliegue.",
     "Cap. 2, sec. 2.5 (auditoria de fuga)", "Prueba ejecutable F3 y F4 del cap. 2; tests/test_ingenieria_variables.py (59 pruebas)."),
    ("Guia E4, sec. 3", "EDA: interpretacion de cada figura y declaracion de los datos (procedencia, licencia, sesgos).",
     "Nuevo cap. 2 con declaracion de datos tipo datasheet y una lectura debajo de cada figura.",
     "Cap. 2, secs. 2.1 y 2.4", "Figuras 2.1 a 2.4 con lectura."),
    ("Guia E4, sec. 3", "Auditoria explicita de fuga de informacion (preprocesamiento, variables que codifican la respuesta, duplicados, mezcla temporal).",
     "Seis pruebas ejecutables antes de congelar resultados.",
     "Cap. 2, sec. 2.5", "Tabla 2.3 (resumen de la auditoria)."),
    ("Guia E4, sec. 3", "Validacion anidada.",
     "E3: CV anidada 5 x 3 en las 140 combinaciones. E4: mismos 5 pliegues externos para los 10 modelos; conjuntos adicionales con 5 x 2 externos y 3 pliegues internos para todos (nunca 80/20).",
     "Cap. 3, sec. 3.2; Cap. 7; protocolo 8.2 y 8.6", "resultados/tabla_maestra.csv; resultados_e4/f5_adicionales/."),
    ("Guia E4, sec. 3", "Implementacion de las tecnicas de optimizacion.",
     "E3: 4 optimizadores con el mismo presupuesto (20 evaluaciones). E4: TPE con 20 evaluaciones para los 10 modelos en los conjuntos adicionales; presupuesto 0 para todos en Home Credit.",
     "Cap. 3, sec. 3.4", "Tabla 3.4 (presupuesto por modelo)."),
    ("Guia E4, sec. 3", "Comparacion estadistica.",
     "SPA, Reality Check y Diebold-Mariano son pruebas de series de tiempo: no se usan para R3. Se usan DeLong, Nadeau-Bengio, Wilcoxon por bloques, Friedman + Nemenyi/Holm, BCa, Cliff, McNemar y bayesiana.",
     "Cap. 6; protocolo 8.4", "resultados_e4/e10_igualdad/pruebas_elar.json; resultados_e4/f7_estadistica/."),
    ("Guia E4, sec. 3", "Interpretabilidad.",
     "El modelo original es aditivo en el logit (cada vista es interpretable); la interpretacion de XGBoost (SHAP) del E3 se mantiene.",
     "Cap. 5 (Persona 1)", "MODEL_CARD.md; cap. 5."),
    # --- Autoauditoria ---
    ("Autoauditoria", "El embudo de variables del E3 excluyo EXT_SOURCE_1, EXT_SOURCE_3 y credit_card_balance por cobertura < 85 %, aunque son muy predictivas.",
     "Enmienda E1 del protocolo: 83 variables nuevas (recuperadas y agregados nuevos), declaradas antes de correr.",
     "Cap. 2, sec. 2.3; protocolo 6.2", "evidencia_ingenieria_variables/catalogo_variables.csv (AUC univariado de EXT_SOURCE_1/3 = 0.67)."),
    ("Autoauditoria", "Fuga minima en IV_VECINOS_MORA_500: el suavizado usaba la tasa global, que incluye la etiqueta de la propia fila (peso ~1e-7).",
     "El suavizado pasa a usar la tasa de los otros pliegues; se verifica por fuerza bruta.",
     "Cap. 2, sec. 2.5 (prueba F4); protocolo 6.2", "tests/test_ingenieria_variables.py; prueba F4 del cap. 2."),
    ("Autoauditoria", "Comparacion desigual: ELAR-R con 83 variables nuevas contra el XGBoost del E3 sin ellas (la ventaja era de las variables, no del algoritmo).",
     "Enmiendas E2 y E3: los 8 y luego 10 modelos con las mismas 528 entradas, los mismos pliegues y la misma prueba. El ranking cambio y se reporta.",
     "Cap. 1, sec. 1.2; Cap. 3, sec. 3.6; Cap. 6", "Tabla 1.1 (ranking antes y despues)."),
    ("Autoauditoria", "En el E3 la estimacion previa de computo fue 5.1 veces menor que el real (87.8 h vs 444.5 h).",
     "En el E4 se mide el ajuste de los modelos lentos antes de estimar y las corridas largas son reanudables por unidad.",
     "Protocolo 8.2; Anexo de reproducibilidad", "resultados_e4/f6_robustez/reentreno.csv (segundos por ajuste)."),
    ("Autoauditoria", "El dashboard no debe reentrenar ni optimizar.",
     "Artefactos serializados (preprocesador + modelo) con metadata.json y prueba de ida y vuelta.",
     "Cap. 3, sec. 3.5", "artifacts/metadata.json; tests/test_artefactos.py (9 pruebas)."),
]
traza = pd.DataFrame(filas, columns=["origen", "observacion", "accion_realizada", "ubicacion_en_el_libro", "evidencia"])
traza.index = pd.RangeIndex(1, len(traza) + 1, name="n")
(RAIZ / "resultados_e4").mkdir(exist_ok=True)
traza.to_csv(RAIZ / "resultados_e4" / "trazabilidad_correcciones.csv", encoding="utf-8")
print(traza["origen"].value_counts().to_string())
traza

origen
Guia E4, sec. 3    7
Autoauditoria      5
Docente, E1        4


,origen,observacion,accion_realizada,ubicacion_en_el_libro,evidencia
n,,,,,
1,"Docente, E1",Un output marcado como ejecucion 67 en una celda que hoy ocupa la posicion 55: no se puede certificar que las cifras salgan del codigo tal como esta.,"Todos los cuadernos se ejecutan de corrido en un kernel nuevo con un script (correr_capitulos.py, nbclient); ninguna cifra se escribe a mano.","Cap. 1, sec. 1.3","Auditoria automatica: contador de ejecucion 1, 2, ..., n en cada cuaderno (tabla 1.2)."
2,"Docente, E1",El cuaderno nunca se ejecuto de corrido en una sola sesion.,Igual que la fila anterior; los calculos costosos se hacen con scripts versionados (experiments/) y los cuadernos leen sus resultados guardados.,"Cap. 1, sec. 1.3; Anexo de reproducibilidad",Tabla 1.2; resultados_e4/*.json con fecha y versiones.
3,"Docente, E1",Rutas absolutas de al menos dos computadores distintos.,Todas las rutas son relativas a la raiz del repositorio (Path.cwd() o config.RAIZ); en las salidas solo se imprime RAIZ.name.,"Cap. 1, sec. 1.3",Tabla 1.2: conteo de rutas absolutas en codigo y salidas = 0.
4,"Docente, E1",Una celda cuyo output no corresponde al codigo mostrado.,Ejecucion completa en kernel nuevo antes de compilar el libro; el libro no se compila con salidas viejas.,"Cap. 1, sec. 1.3",Tabla 1.2 (orden de ejecucion consecutivo).
5,"Guia E4, sec. 3",ETL: transformaciones con parametros aprendidos ajustadas solo dentro de cada pliegue de entrenamiento.,"El preprocesador (imputacion, escala, one-hot) se construye sin ajustar y se ajusta con el entrenamiento de cada pliegue; la variable de vecindario usa codificacion fuera de pliegue.","Cap. 2, sec. 2.5 (auditoria de fuga)",Prueba ejecutable F3 y F4 del cap. 2; tests/test_ingenieria_variables.py (59 pruebas).
6,"Guia E4, sec. 3","EDA: interpretacion de cada figura y declaracion de los datos (procedencia, licencia, sesgos).",Nuevo cap. 2 con declaracion de datos tipo datasheet y una lectura debajo de cada figura.,"Cap. 2, secs. 2.1 y 2.4",Figuras 2.1 a 2.4 con lectura.
7,"Guia E4, sec. 3","Auditoria explicita de fuga de informacion (preprocesamiento, variables que codifican la respuesta, duplicados, mezcla temporal).",Seis pruebas ejecutables antes de congelar resultados.,"Cap. 2, sec. 2.5",Tabla 2.3 (resumen de la auditoria).
8,"Guia E4, sec. 3",Validacion anidada.,E3: CV anidada 5 x 3 en las 140 combinaciones. E4: mismos 5 pliegues externos para los 10 modelos; conjuntos adicionales con 5 x 2 externos y 3 pliegues internos para todos (nunca 80/20).,"Cap. 3, sec. 3.2; Cap. 7; protocolo 8.2 y 8.6",resultados/tabla_maestra.csv; resultados_e4/f5_adicionales/.
9,"Guia E4, sec. 3",Implementacion de las tecnicas de optimizacion.,E3: 4 optimizadores con el mismo presupuesto (20 evaluaciones). E4: TPE con 20 evaluaciones para los 10 modelos en los conjuntos adicionales; presupuesto 0 para todos en Home Credit.,"Cap. 3, sec. 3.4",Tabla 3.4 (presupuesto por modelo).


## 1.2 Resultados recalculados y cambio de ranking

La guía exige reportar si el ranking cambia tras las correcciones. Cambió. En el E3 cada modelo usó sus propias entradas (las del embudo
del E3). Con la enmienda E1, ELAR-R con las 83 variables nuevas superaba al XGBoost del E3 (0.7899 frente a 0.7754 en la prueba). Esa
comparación era desigual. Al darles a todos las mismas 528 entradas (enmienda E3, sec. 6.5 del protocolo) XGBoost vuelve a ganar y ELAR-R
queda tercero de 10. La tabla 1.1 pone los dos rankings lado a lado; todas las tablas y pruebas de los capítulos 3, 6 y 7 usan la versión corregida.

In [3]:
# Tabla 1.1: ranking del E3 (AUC externo medio de la mejor combinacion de cada familia, entradas del E3)
# frente al ranking corregido del E4 (10 modelos, mismas 528 entradas, AUC fuera de muestra con las 5 validaciones externas unidas).
e3 = pd.read_csv(RAIZ / "evidencia_ingenieria_variables" / "ganadores_por_familia.csv", encoding="utf-8-sig")
e3 = e3[e3["tarea"] == "clasificacion"].set_index("familia")["auc_externo_media"]
e4 = json.loads((RAIZ / "resultados_e4" / "e10_igualdad" / "pruebas_elar.json").read_text(encoding="utf-8"))
nombre = {"elar": "ELAR-R (original)", "xgboost": "XGBoost", "catboost": "CatBoost", "lightgbm": "LightGBM",
          "logistica": "Regresion logistica", "svm": "SVM", "random_forest": "Random forest", "arbol": "Arbol",
          "knn": "KNN", "naive_bayes": "Naive Bayes"}
t = pd.DataFrame({"AUC E3 (entradas propias)": e3, "AUC E4 igualdad (OOF)": pd.Series(e4["auc_oof"]),
                  "AUC E4 igualdad (prueba)": pd.Series(e4["auc_prueba_final"])})
t.loc["elar", "AUC E3 (entradas propias)"] = np.nan          # ELAR-R no existia en el E3
t["puesto E3"] = t["AUC E3 (entradas propias)"].rank(ascending=False).astype("Int64")
t["puesto E4"] = t["AUC E4 igualdad (OOF)"].rank(ascending=False).astype(int)
t = t.sort_values("puesto E4").rename(index=nombre)
t.round(4)

,AUC E3 (entradas propias),AUC E4 igualdad (OOF),AUC E4 igualdad (prueba),puesto E3,puesto E4
XGBoost,0.7721,0.7877,0.7902,1,1
CatBoost,NaN,0.7843,0.7898,<NA>,2
ELAR-R (original),NaN,0.7838,0.7871,<NA>,3
LightGBM,NaN,0.7835,0.7867,<NA>,4
Regresion logistica,0.7576,0.7738,0.7765,3,5
SVM,0.7578,0.7725,0.7753,2,6
Random forest,0.7497,0.7656,0.7694,4,7
Arbol,0.6965,0.7197,0.7253,5,8
KNN,0.6887,0.6992,0.7045,6,9
Naive Bayes,0.6464,0.6585,0.6579,7,10


**Lectura de la tabla 1.1.** Con las mismas entradas todos los modelos suben respecto al E3 (las variables recuperadas ayudan a todos),
y el orden de los siete ganadores de familia del E3 se conserva salvo un intercambio: XGBoost primero; luego SVM y logística, que estaban
empatados en el E3 (0.7578 y 0.7576) y en el E4 cambian de lugar; después random forest, árbol, KNN y Naive Bayes. ELAR-R entra tercero, detrás de XGBoost y CatBoost y prácticamente empatado con LightGBM. Este es el resultado desfavorable que la
guía pide no ocultar: la ventaja que mostraba ELAR-R en la enmienda E1 venía de la ingeniería de variables y no del algoritmo.

## 1.3 Auditoría automática de los cuadernos

La celda siguiente revisa todos los cuadernos del libro (`*.ipynb` en la raíz) y comprueba, a partir del propio JSON de cada uno,
lo que observó el docente en el E1:

- **orden de ejecución:** los contadores de las celdas de código ejecutadas deben ser 1, 2, …, n, en el orden en que aparecen;
- **celdas sin ejecutar:** ninguna celda de código con contenido puede quedar sin contador;
- **rutas absolutas:** se buscan patrones como `C:\Users`, `/home/<usuario>` o `/Users/<usuario>` en el código y en las salidas.

Este cuaderno se excluye a sí mismo (se está ejecutando mientras se audita) y por eso se ejecuta el último en `correr_capitulos.py`.

In [4]:
PATRON_ABS = re.compile(r"([A-Za-z]:\\\\?Users\\\\?|/home/[a-z_][\w-]*/|/Users/[\w.-]+/|/mnt/user-data/)")


def texto_salidas(celda):
    # Une el texto de todas las salidas de una celda (las imagenes no se revisan).
    partes = []
    for o in celda.get("outputs", []):
        partes.append("".join(o.get("text", "")))
        for clave, valor in o.get("data", {}).items():
            if clave.startswith("text/"):
                partes.append("".join(valor) if isinstance(valor, list) else str(valor))
        partes.append("".join(o.get("traceback", [])))
    return "\n".join(partes)


def auditar(ruta):
    nb = json.loads(ruta.read_text(encoding="utf-8"))
    codigo = [c for c in nb["cells"] if c["cell_type"] == "code" and "".join(c["source"]).strip()]
    cont = [c.get("execution_count") for c in codigo]
    ejecutadas = [x for x in cont if x is not None]
    abs_codigo = sum(len(PATRON_ABS.findall("".join(c["source"]))) for c in codigo)
    abs_salida = sum(len(PATRON_ABS.findall(texto_salidas(c))) for c in codigo)
    errores = sum(o.get("output_type") == "error" for c in codigo for o in c.get("outputs", []))
    return {"cuaderno": ruta.name, "celdas_codigo": len(codigo), "sin_ejecutar": cont.count(None),
            "orden_1_a_n": ejecutadas == list(range(1, len(ejecutadas) + 1)) and len(ejecutadas) > 0,
            "rutas_abs_codigo": abs_codigo, "rutas_abs_salidas": abs_salida, "celdas_con_error": errores}


propio = "01_correcciones.ipynb"
COLS = ["cuaderno", "celdas_codigo", "sin_ejecutar", "orden_1_a_n", "rutas_abs_codigo", "rutas_abs_salidas", "celdas_con_error"]
auditoria = pd.DataFrame([auditar(r) for r in sorted(RAIZ.glob("*.ipynb")) if r.name != propio], columns=COLS)
auditoria["cumple"] = (auditoria["sin_ejecutar"].eq(0) & auditoria["orden_1_a_n"]
                       & auditoria[["rutas_abs_codigo", "rutas_abs_salidas", "celdas_con_error"]].sum(axis=1).eq(0))
auditoria.to_csv(RAIZ / "resultados_e4" / "auditoria_cuadernos.csv", index=False)
print(f"Cuadernos auditados: {len(auditoria)}; cumplen: {int(auditoria['cumple'].sum())}")
auditoria

Cuadernos auditados: 15; cumplen: 15


,cuaderno,celdas_codigo,sin_ejecutar,orden_1_a_n,rutas_abs_codigo,rutas_abs_salidas,celdas_con_error,cumple
0,01_datos_y_linea_base.ipynb,77,0,True,0,0,0,True
1,02_clasificacion.ipynb,17,0,True,0,0,0,True
2,02_datos.ipynb,13,0,True,0,0,0,True
3,03_benchmarks.ipynb,9,0,True,0,0,0,True
4,03_regresion.ipynb,14,0,True,0,0,0,True
5,04_optimizacion_hiperparametros.ipynb,14,0,True,0,0,0,True
6,05_optimizacion_computacional.ipynb,14,0,True,0,0,0,True
7,06_evaluacion_clasificacion.ipynb,10,0,True,0,0,0,True
8,06_validacion.ipynb,8,0,True,0,0,0,True
9,07_evaluacion_ampliada.ipynb,14,0,True,0,0,0,True


**Lectura de la tabla 1.2.** Un cuaderno cumple si todas sus celdas de código se ejecutaron en orden consecutivo, ninguna quedó con error
y no aparece ninguna ruta absoluta ni en el código ni en las salidas. La tabla se genera al compilar el libro, de modo que refleja el
estado real de los archivos entregados y no una afirmación del equipo. Si alguna fila aparece con `cumple = False`, el cuaderno se vuelve a
ejecutar con `correr_capitulos.py` antes de entregar.